Yes—filtering by document name first is a good idea, especially when each deal has 2–10 documents of roughly 200 pages. But make it the first routing layer, not the only decision rule: filenames are often inconsistent, and an important drawdown clause may be in an “Amended and Restated Facility Agreement” or a schedule with an unhelpful name.

The scalable approach is:

Deal files → Filename triage → Page-level text extraction → Targeted clause retrieval → Evidence extraction

You do not need to load 2,000 pages into an LLM per deal. Extract text once, search locally, and only inspect the few candidate pages.

#### Why filename filtering helps
If each deal has 10 documents × 200 pages, that is around 2,000 pages per deal. Most of those pages are unlikely to contain the actual operative drawdown notice condition.

Document names can help you rank likely sources before processing:

| Filename / document type                                | Likelihood of operative drawdown clause | Action                                                    |
| ------------------------------------------------------- | --------------------------------------- | --------------------------------------------------------- |
| Facility Agreement, Loan Agreement, Credit Agreement    | Very high                               | Process first                                             |
| Amended, Restated, Supplemental, Variation, Amendment   | Very high                               | Process first; may override original agreement            |
| Term Sheet, Offer Letter                                | Medium                                  | Process, but treat as indicative unless legally operative |
| Schedule, Annex, Appendix                               | Medium to high                          | Process if tied to utilisation, notice, or facility terms |
| Utilisation Request, Drawdown Notice, Borrowing Request | High for format/process                 | Process; may be a template rather than the binding clause |
| Conditions Precedent                                    | Medium                                  | Search, because it may include pre-drawdown requirements  |
| Security Agreement, Guarantee, Account Charge           | Low                                     | Usually deprioritise                                      |
| Board Resolution, Certificate, KYC, Corporate Documents | Very low                                | Skip initially                                            |
| Fee Letter, Invoice, Tax, Audit, Financial Statements   | Very low                                | Skip initially                                            |

The key nuance is that a file called Drawdown Notice Template.pdf may contain the notice format, but not necessarily the contractual requirement for “three Business Days’ prior notice.” The authoritative rule is more likely to be in the agreement or its amendments.

#### Use a two-pass strategy

##### Pass 1: Fast filename triage
Score documents from their filename and document type. Use this to decide processing order and where to spend more effort.

For example:

In [ ]:
from pathlib import Path
import re


DOCUMENT_RULES = {
    "high_priority": {
        "patterns": [
            r"\bfacility\b.*\bagreement\b",
            r"\bloan\b.*\bagreement\b",
            r"\bcredit\b.*\bagreement\b",
            r"\bamended\b",
            r"\brestated\b",
            r"\bamendment\b",
            r"\bsupplemental\b",
            r"\bvariation\b",
            r"\butilisation\b",
            r"\butilization\b",
            r"\bdrawdown\b",
            r"\bborrowing\b.*\brequest\b",
            r"\bdrawdown\b.*\bnotice\b"
        ],
        "score": 10
    },
    "medium_priority": {
        "patterns": [
            r"\bterm\b.*\bsheet\b",
            r"\boffer\b.*\bletter\b",
            r"\bschedule\b",
            r"\bannex\b",
            r"\bappendix\b",
            r"\bconditions?\b.*\bprecedent\b",
            r"\bcp\b"
        ],
        "score": 5
    },
    "low_priority": {
        "patterns": [
            r"\bsecurity\b",
            r"\bguarantee\b",
            r"\bcharge\b",
            r"\bkyc\b",
            r"\bboard\b.*\bresolution\b",
            r"\bcertificate\b",
            r"\baudit\b",
            r"\bfinancial\b.*\bstatement\b",
            r"\binvoice\b",
            r"\bfee\b.*\bletter\b"
        ],
        "score": -5
    }
}


def score_filename(file_path: str) -> dict:
    filename = Path(file_path).stem.lower()
    score = 0
    matched_rules = []

    for priority, config in DOCUMENT_RULES.items():
        for pattern in config["patterns"]:
            if re.search(pattern, filename, flags=re.IGNORECASE):
                score += config["score"]
                matched_rules.append({
                    "priority_group": priority,
                    "pattern": pattern
                })

    if score >= 10:
        priority = "high"
    elif score >= 1:
        priority = "medium"
    else:
        priority = "low"

    return {
        "filename": Path(file_path).name,
        "priority": priority,
        "filename_score": score,
        "matched_rules": matched_rules
    }

In [ ]:
# Example
files = [
    "DEAL001_Facility_Agreement.pdf",
    "DEAL001_Schedule_2_Conditions_Precedent.pdf",
    "DEAL001_Security_Agreement.pdf",
    "DEAL001_Amendment_Letter_2025.pdf",
    "DEAL001_Board_Resolution.pdf"
]

for file in files:
    print(score_filename(file))

Expected prioritisation:

DEAL001_Facility_Agreement.pdf      → high
DEAL001_Amendment_Letter_2025.pdf   → high
DEAL001_Schedule_2_Conditions...    → medium
DEAL001_Security_Agreement.pdf      → low
DEAL001_Board_Resolution.pdf        → low

This lets you immediately avoid spending OCR or LLM resources on low-value files unless needed.

Do not filter out files permanently
Use filenames to prioritise, not to exclude every “irrelevant” file forever.

A safer workflow per Deal_ID is:
1. Process all high-priority documents first.
2. Search each page for drawdown-related concepts.
3. If you find high-confidence evidence, stop or mark the deal as provisionally resolved.
4. If no result appears, expand to medium-priority documents.
5. Only search low-priority files if:
    - No operative agreement is present.
    - The deal documentation is incomplete.
    - The high-priority documents are scanned/unreadable.
    - A reviewer requests a broader search.
This gives you a large speed gain without creating an avoidable false negative.

Process 200-page PDFs efficiently
For your use case, extract the text once and store it locally. You then search the saved output repeatedly without reopening and reprocessing every PDF.

PyMuPDF supports page-level extraction, including text and metadata, so it is suitable for retaining page references in your evidence output.

Store one row per page
Instead of treating a 200-page agreement as one huge document, create a local page index:

| Deal_ID  | File                   | File priority | Page | Extracted text | Extraction status |
| -------- | ---------------------- | ------------- | ---- | -------------- | ----------------- |
| DEAL-001 | Facility Agreement.pdf | High          | 1    | …              | Text              |
| DEAL-001 | Facility Agreement.pdf | High          | 2    | …              | Text              |
| DEAL-001 | Facility Agreement.pdf | High          | 3    | …              | Text              |
| DEAL-001 | Amendment Letter.pdf   | High          | 1    | …              | Text              |

Save it to Parquet if possible, rather than a massive CSV:


In [ ]:
from pathlib import Path
import fitz
import pandas as pd


def extract_pdf_to_page_index(
    deal_id: str,
    file_path: str,
    filename_score: int,
    priority: str
) -> list[dict]:
    rows = []
    file_name = Path(file_path).name

    with fitz.open(file_path) as pdf:
        for page_index, page in enumerate(pdf):
            text = page.get_text("text", sort=True).strip()

            rows.append({
                "deal_id": deal_id,
                "file_path": str(file_path),
                "file_name": file_name,
                "file_priority": priority,
                "filename_score": filename_score,
                "page_number": page_index + 1,
                "page_text": text,
                "text_length": len(text),
                "needs_ocr": len(text) < 50
            })

    return rows


all_page_rows = []

for _, item in deal_document_map.iterrows():
    file_path = item["file_path"]
    deal_id = item["Deal_ID"]

    file_routing = score_filename(file_path)

    if file_routing["priority"] == "low":
        continue

    page_rows = extract_pdf_to_page_index(
        deal_id=deal_id,
        file_path=file_path,
        filename_score=file_routing["filename_score"],
        priority=file_routing["priority"]
    )

    all_page_rows.extend(page_rows)

page_index_df = pd.DataFrame(all_page_rows)

page_index_df.to_parquet(
    "output/page_index.parquet",
    index=False
)

On later runs:

`page_index_df = pd.read_parquet("output/page_index.parquet")`

This separation is important:
- Ingestion runs once, or only for new/changed files.
- Search rules can be improved and rerun in seconds.
- LLM validation is run only on selected passages, not every page.

#### Use page-level scoring before chunking
For your specific clause, the fastest first-stage retriever may be page-level scoring rather than embeddings.

Search each page for a combination of terms:
- Drawdown / utilisation / borrowing / advance
- Request / notice / delivery
- Business Day / Working Day
- “Not later than,” “at least,” “prior to,” “before”

In [ ]:
import re


DRAW_DOWN_PATTERNS = [
    r"\bdrawdown\b",
    r"\bdraw down\b",
    r"\butilisation\b",
    r"\butilization\b",
    r"\butilisation request\b",
    r"\butilization request\b",
    r"\bborrowing request\b",
    r"\bloan request\b",
    r"\badvance request\b",
]

NOTICE_PATTERNS = [
    r"\bnotice\b",
    r"\bprior written notice\b",
    r"\bnot later than\b",
    r"\bat least\b",
    r"\bprior to\b",
    r"\bbefore\b",
]

DAY_PATTERNS = [
    r"\bbusiness days?\b",
    r"\bworking days?\b",
]


def score_page_for_drawdown_notice(page_text: str) -> dict:
    text = page_text.lower()

    drawdown_hits = sum(
        bool(re.search(pattern, text, flags=re.IGNORECASE))
        for pattern in DRAW_DOWN_PATTERNS
    )

    notice_hits = sum(
        bool(re.search(pattern, text, flags=re.IGNORECASE))
        for pattern in NOTICE_PATTERNS
    )

    day_hits = sum(
        bool(re.search(pattern, text, flags=re.IGNORECASE))
        for pattern in DAY_PATTERNS
    )

    score = (
        drawdown_hits * 5
        + notice_hits * 3
        + day_hits * 5
    )

    likely_clause = (
        drawdown_hits > 0
        and notice_hits > 0
        and day_hits > 0
    )

    return {
        "drawdown_hits": drawdown_hits,
        "notice_hits": notice_hits,
        "business_day_hits": day_hits,
        "page_score": score,
        "likely_clause": likely_clause
    }

In [ ]:
# Apply it
scores = page_index_df["page_text"].apply(score_page_for_drawdown_notice)

score_df = pd.DataFrame(scores.tolist())

page_index_df = pd.concat(
    [page_index_df, score_df],
    axis=1
)

candidate_pages_df = (
    page_index_df[
        (page_index_df["likely_clause"])
        | (page_index_df["page_score"] >= 8)
    ]
    .sort_values(
        ["deal_id", "filename_score", "page_score"],
        ascending=[True, False, False]
    )
)

From 2,000 pages for a deal, you may end up with only 3–20 pages worth inspecting or passing into a more advanced extraction step.

Search for headings first
Formal loan agreements often have predictable headings, even if wording differs. A 200-page agreement could have the relevant clause in a section such as:
- Utilisation
- Utilisations
- Conditions of Utilisation
- Utilisation Requests
- Drawdown
- Borrowing Requests
- Request for a Loan
- Notice of Drawdown
- Availability Period
- Loans
- Advances

This is more efficient than indiscriminately chunking all pages.

In [ ]:
HEADING_PATTERNS = [
    r"^\s*\d+\.?\s+utilisations?\s*$",
    r"^\s*\d+\.?\s+drawdowns?\s*$",
    r"^\s*\d+\.?\s+utilisation requests?\s*$",
    r"^\s*\d+\.?\s+utilization requests?\s*$",
    r"^\s*\d+\.?\s+borrowing requests?\s*$",
    r"^\s*\d+\.?\s+conditions of utilisation\s*$",
    r"^\s*\d+\.?\s+conditions of utilization\s*$",
]

If a page contains a heading like 5. Utilisation Requests, include:

- That page
- The next 1–3 pages
- The previous page, in case the heading is split by a page break

This “section expansion” is very effective for legal documents because the notice period may appear in the clause immediately after the heading.

#### Add a document-level stop rule
Since you might have 10 files per deal, define a practical stop rule.

Example decision hierarchy
- Search the latest amended/restated facility agreement.
- Search the original facility/loan/credit agreement.
- Search schedules and appendices relevant to utilisation.
- Search standalone borrowing/drawdown request templates.
- Search conditions precedent documents.
- Search all remaining files only if no evidence is found.

For every match, track whether the source is binding:

| Source type                | How to interpret                                          |
| -------------------------- | --------------------------------------------------------- |
| Amended/restated agreement | Usually highest priority, subject to legal review         |
| Facility/loan agreement    | Primary contractual source                                |
| Amendment / variation      | May replace or change the original period                 |
| Schedule                   | May define facility/tranche-specific mechanics            |
| Drawdown request template  | Useful confirmation, but may not establish the legal rule |
| Term sheet / offer letter  | May be preliminary or superseded                          |
| Email / operational note   | Context only; do not treat as contractual source          |

In [ ]:
# You can automate an initial ranking:
SOURCE_PRECEDENCE = {
    "amended_restated_agreement": 1,
    "amendment_variation": 2,
    "facility_loan_credit_agreement": 3,
    "relevant_schedule_annex": 4,
    "drawdown_notice_template": 5,
    "term_sheet_offer_letter": 6,
    "other": 99
}

A better scoring model
Use both filename relevance and content relevance:

Final score=0.25×Filename score+0.75×Page content score

Content should carry more weight because filenames are unreliable. If a file has a boring name such as Final signed version.pdf, strong text evidence should still rank it highly.



In [ ]:
page_index_df["final_score"] = (
    0.25 * page_index_df["filename_score"]
    + 0.75 * page_index_df["page_score"]
)

top_candidates = (
    page_index_df
    .sort_values(
        ["deal_id", "final_score"],
        ascending=[True, False]
    )
    .groupby("deal_id")
    .head(10)
)

#### For scanned PDFs
Do not OCR all 2,000 pages by default. That will be the slowest and most expensive local step.

Instead:
- Extract native text with PyMuPDF.
- Mark pages with very little text, such as fewer than 50 characters.
- OCR only:
    - High-priority documents, and
    - Pages in suspected target sections, or
    - Entire high-priority documents only when almost all pages are scanned.

OCRmyPDF can add a searchable OCR text layer to scanned PDFs; it also supports rotation correction and deskewing, which can improve text extraction from skewed scans.

Example bash command:

`ocrmypdf --skip-text --rotate-pages --deskew input.pdf output_searchable.pdf`

Then run PyMuPDF against output_searchable.pdf.

#### Where RAG fits later

For this exact task, start with filename triage + page-level rules. It will likely work better than immediately building full RAG because you already know the target concept: drawdown notice and business-day lead time.

Add local semantic retrieval only if you observe missed clauses caused by varied legal wording. A lightweight local BM25 retriever can rank text passages by matching relevant query terms, and rank_bm25 provides a Python implementation for this purpose.

A sensible progression is:
- Filename routing.
- Native text extraction with page metadata.
- OCR only when needed.
- Page scoring with legal-term patterns.
- Passage extraction around matching clauses.
- Regex for business-day values.
- LLM JSON validation for the top passages.
- Hybrid BM25 + embedding search if recall is insufficient.

#### My recommendation for your project

Since you already have experience with local RAG and have found basic PDF loaders suboptimal, avoid PyPDFDirectoryLoader as the core of this process. Use PyMuPDF directly, retain Deal_ID → filename → page number → text metadata, and treat filename filtering as a prioritisation layer.

Your minimum viable version should produce one Excel tab like this:

| Deal_ID  | File name                      | File priority | Page | Candidate clause | Business days found | Evidence                                                 | Confidence | Review   |
| -------- | ------------------------------ | ------------- | ---- | ---------------- | ------------------- | -------------------------------------------------------- | ---------- | -------- |
| DEAL-001 | Amended Facility Agreement.pdf | High          | 34   | Yes              | 3                   | “...not later than three Business Days before...”        | High       | Complete |
| DEAL-002 | Loan Agreement.pdf             | High          | 19   | Yes              | 2                   | “...two Business Days’ prior written notice...”          | High       | Complete |
| DEAL-003 | Drawdown Notice Template.pdf   | High          | 1    | Possibly         | —                   | Template found; no binding timing requirement identified | Medium     | Review   |
| DEAL-004 | All high-priority documents    | —             | —    | Not identified   | —                   | No candidate found after targeted search                 | Low        | Review   |